In [2]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.13.15
Folder: content
numpy - ok
pandas - ok
sklearn - ok


# Build the dataset

In [3]:
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data/delivery_times.csv


# Loads the data, separates features (X) from the target (y), and does an 80/20 train/test split.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


# Define a function that trains a model, measures its error on both the training data and the test data, and computes the gap between them.

In [5]:
from sklearn.metrics import mean_absolute_error

def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

# Model 1: LinearRegression

In [6]:
from sklearn.linear_model import LinearRegression

linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


# Model 2: decision tree, no limit

In [7]:
from sklearn.tree import DecisionTreeRegressor

wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


# Model 3: shallow tree (depth 4)

In [8]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


# Model 4: RandomForest

In [9]:
from sklearn.ensemble import RandomForestRegressor

forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 1.0 test 2.39 gap 1.4


In [10]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


# Cross-validation - Trains and tests each model 5 times on 5 different data splits and averages the error

In [11]:
from sklearn.model_selection import cross_val_score

def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()

cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03
DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


# Tries several tree depths (2, 3, 4, 6, 8, None), scores each with cross-validation, and finds the best one.

In [12]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)

best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.478}
best depth: 8


# Ranks the three model

In [13]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


# To Do: Train Logistic Regression, Decision Tree, and Random Forest classifiers on the Breast Cancer dataset, compare their train/test accuracy, use cross-validation to find the best tree depth, and report the confusion matrix and classification metrics to identify the best-performing model. https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

In [20]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
print("Logistic Regression Train Accuracy:", accuracy_score(y_train, lr.predict(X_train)))
print("Logistic Regression Test Accuracy:", accuracy_score(y_test, lr.predict(X_test)))
print(confusion_matrix(y_test, lr.predict(X_test)))
print(classification_report(y_test, lr.predict(X_test)))

Logistic Regression Train Accuracy: 0.9582417582417583
Logistic Regression Test Accuracy: 0.956140350877193
[[39  4]
 [ 1 70]]
              precision    recall  f1-score   support

           0       0.97      0.91      0.94        43
           1       0.95      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [22]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("Decision Tree Train Accuracy:", accuracy_score(y_train, dt.predict(X_train)))
print("Decision Tree Test Accuracy:", accuracy_score(y_test, dt.predict(X_test)))

Decision Tree Train Accuracy: 1.0
Decision Tree Test Accuracy: 0.9473684210526315


In [23]:
depths = [2, 3, 4, 5, 6, 7, 8, None]
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(clf, X_train, y_train, cv=5)
    print("Depth:", d, "CV Mean Accuracy:", scores.mean())

Depth: 2 CV Mean Accuracy: 0.9186813186813186
Depth: 3 CV Mean Accuracy: 0.9318681318681319
Depth: 4 CV Mean Accuracy: 0.923076923076923
Depth: 5 CV Mean Accuracy: 0.9252747252747252
Depth: 6 CV Mean Accuracy: 0.9164835164835164
Depth: 7 CV Mean Accuracy: 0.9186813186813186
Depth: 8 CV Mean Accuracy: 0.9164835164835164
Depth: None CV Mean Accuracy: 0.9164835164835164


In [24]:
best_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
best_dt.fit(X_train, y_train)
print(confusion_matrix(y_test, best_dt.predict(X_test)))
print(classification_report(y_test, best_dt.predict(X_test)))

[[39  4]
 [ 2 69]]
              precision    recall  f1-score   support

           0       0.95      0.91      0.93        43
           1       0.95      0.97      0.96        71

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



In [25]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
print("Random Forest Train Accuracy:", accuracy_score(y_train, rf.predict(X_train)))
print("Random Forest Test Accuracy:", accuracy_score(y_test, rf.predict(X_test)))
print(confusion_matrix(y_test, rf.predict(X_test)))
print(classification_report(y_test, rf.predict(X_test)))

Random Forest Train Accuracy: 1.0
Random Forest Test Accuracy: 0.9649122807017544
[[40  3]
 [ 1 70]]
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

